[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/09_attention_advanced.ipynb)

# 09. Advanced attention — sparse connectivity and recurrent alternatives

이전 `top-k sparse attention`은 **dense QKᵀ 전체를 먼저 계산한 뒤 top-k만 남겼기 때문에 계산량 관점에서는 sparse attention이 아니었다**. 또 DeltaNet section은 state update 한 번만 있어 sequence model의 recurrent read/write가 보이지 않았다.

이번 버전에서는 local attention, 실제로 선택된 block만 QK를 계산하는 sparse example, sigmoid attention의 normalization 차이, 그리고 DeltaNet의 sequence-level recurrent fast-weight update를 분리한다.


In [ ]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Sliding-window attention

각 query가 가까운 `w`개 token만 읽도록 connectivity를 제한한다. dense score matrix를 conceptual mask로 표현할 수 있지만 efficient kernel에서는 허용된 band만 계산해야 실제 시간/메모리 이득이 난다.


In [ ]:
sequence_length = 8
window_radius = 1
head_dim = 4

q = torch.randn(1, 1, sequence_length, head_dim, device=device)
k = torch.randn_like(q)
v = torch.randn_like(q)

positions = torch.arange(sequence_length, device=device)
allowed = (
    positions[:, None] - positions[None, :]
).abs() <= window_radius

local_output = F.scaled_dot_product_attention(
    q,
    k,
    v,
    attn_mask=allowed,
)

print("local connectivity:\n", allowed.int())
print("output:", local_output.shape)


## 2. Selected-block sparse attention without full token-level QK

진짜 sparse attention의 핵심은 **선택 전에 full token×token score를 계산하지 않는 것**이다. 아래 toy example은 block summary끼리 coarse score를 계산해 relevant key blocks를 고르고, 그 block 내부 token에 대해서만 fine QK를 계산한다.

이것은 특정 DeepSeek NSA 구현을 그대로 복제한 것이 아니라, dense-then-topk와 native sparse compute의 차이를 보여주는 최소 구조다.


In [ ]:
sequence_length = 16
block_size = 4
num_blocks = sequence_length // block_size
head_dim = 8

q_tokens = torch.randn(sequence_length, head_dim, device=device)
k_tokens = torch.randn(sequence_length, head_dim, device=device)
v_tokens = torch.randn(sequence_length, head_dim, device=device)

q_blocks = q_tokens.view(num_blocks, block_size, head_dim)
k_blocks = k_tokens.view(num_blocks, block_size, head_dim)
v_blocks = v_tokens.view(num_blocks, block_size, head_dim)

q_summary = q_blocks.mean(dim=1)
k_summary = k_blocks.mean(dim=1)

coarse_scores = (
    q_summary @ k_summary.T
) / math.sqrt(head_dim)
selected_key_block = coarse_scores.argmax(dim=-1)

sparse_outputs = []
fine_score_count = 0

for query_block_id in range(num_blocks):
    key_block_id = int(selected_key_block[query_block_id])

    q_block = q_blocks[query_block_id]
    k_block = k_blocks[key_block_id]
    v_block = v_blocks[key_block_id]

    fine_scores = (
        q_block @ k_block.T
    ) / math.sqrt(head_dim)
    fine_weights = fine_scores.softmax(dim=-1)
    sparse_outputs.append(fine_weights @ v_block)

    fine_score_count += fine_scores.numel()

sparse_output = torch.cat(sparse_outputs, dim=0)
dense_score_count = sequence_length * sequence_length

print("selected key blocks:", selected_key_block)
print("fine token scores computed:", fine_score_count)
print("dense token scores would be:", dense_score_count)
print("sparse output:", sparse_output.shape)


## 3. Sigmoid attention is not row-normalized softmax attention

softmax는 한 query row의 weights 합을 1로 강제하지만 sigmoid는 각 edge를 독립적으로 gate한다. 그래서 raw sigmoid weight의 총 크기가 sequence length에 따라 커질 수 있어 실제 설계에서는 scaling/normalization choice가 중요하다.


In [ ]:
q = torch.randn(1, 2, 6, 4, device=device)
k = torch.randn_like(q)
v = torch.randn_like(q)

scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))

softmax_weights = scores.softmax(dim=-1)
sigmoid_weights = torch.sigmoid(scores)

softmax_output = softmax_weights @ v
sigmoid_output = sigmoid_weights @ v

print("softmax row sums:", softmax_weights.sum(dim=-1))
print("sigmoid row sums:", sigmoid_weights.sum(dim=-1))
print("softmax output norm:", softmax_output.norm().item())
print("sigmoid output norm:", sigmoid_output.norm().item())


## 4. DeltaNet: recurrent fast-weight memory over a full sequence

DeltaNet 계열은 `N×N` attention matrix를 저장하는 대신 fixed-size matrix state `S_t`를 유지한다. key `k_t`로 현재 memory prediction `S_{t-1}k_t`를 읽고, prediction error `v_t-S_{t-1}k_t`만큼 rank-1 delta-rule update를 한다. query `q_t`는 갱신된 state에서 output을 읽는다.


In [ ]:
def delta_net_sequence(q, k, v, beta):
    sequence_length, key_dim = k.shape
    value_dim = v.size(-1)

    state = torch.zeros(
        value_dim,
        key_dim,
        device=q.device,
    )
    outputs = []

    for time_index in range(sequence_length):
        key_t = F.normalize(k[time_index], dim=0)
        query_t = F.normalize(q[time_index], dim=0)
        value_t = v[time_index]
        beta_t = beta[time_index]

        predicted_value = state @ key_t
        error = value_t - predicted_value

        state = (
            state
            + beta_t * torch.outer(error, key_t)
        )

        output_t = state @ query_t
        outputs.append(output_t)

    return torch.stack(outputs), state


sequence_length = 7
key_dim = 4
value_dim = 5

q = torch.randn(sequence_length, key_dim, device=device)
k = torch.randn(sequence_length, key_dim, device=device)
v = torch.randn(sequence_length, value_dim, device=device)
beta = torch.sigmoid(torch.randn(sequence_length, device=device))

outputs, final_state = delta_net_sequence(q, k, v, beta)

print("sequence outputs:", outputs.shape)
print("fixed recurrent state:", final_state.shape)


## 5. KDA/Gated DeltaNet are not identical to plain DeltaNet

Kimi Delta Attention와 Gated DeltaNet 계열은 forgetting/decay와 gating을 추가해 state memory dynamics를 바꾼다. 여기서는 검증된 plain delta-rule core까지만 구현하고 KDA 전체를 한 줄 state update로 가장하지 않는다.


## References and provenance

**Sliding-window attention** — Longformer/Mistral 계열의 local connectivity를 참조했다.

**Sparse attention** — block summary로 routing한 뒤 selected token blocks에서만 fine QK를 계산하는 toy를 사용했다. 이것은 특정 NSA/DSA 구현을 그대로 재현한다고 주장하지 않는다. 핵심 목적은 `dense QK → top-k`가 sparse compute가 아니라는 점을 분명히 하는 것이다.

**Sigmoid attention** — softmax normalization을 독립 sigmoid gates로 바꿀 때 row-normalization 성질이 사라지는 핵심 차이를 보여준다.

**DeltaNet** — delta-rule fast-weight memory의 recurrent read, prediction-error write, fixed matrix state를 full sequence loop로 반영했다. KDA/Gated DeltaNet의 decay/gating은 별도 확장으로 구분한다.
